In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, RocCurveDisplay
)
# Fetch dataset from UCI ML Repo
dataset = fetch_ucirepo(id=350)
X = dataset.data.features
y = dataset.data.targets.squeeze().astype(int)  # Convert Series to int for classification
# Clean column names if needed
X.columns = X.columns.str.strip()
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# Baseline Model
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)
y_pred_base = rf_baseline.predict(X_test)
y_proba_base = rf_baseline.predict_proba(X_test)[:, 1]
# Evaluation - Baseline
print("\n📊 Baseline Performance")
print("Accuracy:", round(accuracy_score(y_test, y_pred_base), 4))
print("AUC Score:", round(roc_auc_score(y_test, y_proba_base), 4))
print(classification_report(y_test, y_pred_base))
# 🔹 Hyperparameter Tuning
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt']
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=3, scoring='roc_auc', verbose=1, n_jobs=-1
)
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_
# Evaluation - Tuned
y_pred_tuned = best_rf.predict(X_test)
y_proba_tuned = best_rf.predict_proba(X_test)[:, 1]
print("\n✅ Tuned Model Parameters:", grid.best_params_)
print("Accuracy:", round(accuracy_score(y_test, y_pred_tuned), 4))
print("AUC Score:", round(roc_auc_score(y_test, y_proba_tuned), 4))
print(classification_report(y_test, y_pred_tuned))
# 🔍 Confusion Matrix Comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, y_pred_base), annot=True, fmt='d', cmap='Blues')
plt.title("Baseline Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, y_pred_tuned), annot=True, fmt='d', cmap='Greens')
plt.title("Tuned Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.show()
# 📈 ROC Curve
plt.figure(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_proba_base, name="Baseline RF")
RocCurveDisplay.from_predictions(y_test, y_proba_tuned, name="Tuned RF")
plt.plot([0, 1], [0, 1], 'k--')
plt.title("ROC Curve Comparison")
plt.grid(True)
plt.tight_layout()
plt.show()